[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L10.ipynb)

# Lecture 10 — Loss Functions & Optimizers

*The number that says how wrong you are — and the engine that does something about it.*

**PyTorch from Ground Up · Volume I · Chapters 16–17** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Choose a loss from the task: MSE for a number, cross-entropy for a class.
- Compute MSE and cross-entropy by hand and match PyTorch’s output.
- Avoid the logits trap: never softmax before `CrossEntropyLoss`.
- Explain `reduction` and why `mean` is the default.
- Trace the evolution GD → SGD → momentum → Adam and say what each one fixes.
- Use the two-line optimizer API, pick a learning rate, and add weight decay.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · What the loss decides

Predict → **measure the error** → find the slope → step downhill. Today: the measuring. Nothing today is new syntax — it is the meaning behind lines you already type.

<sub>Source: lecture slides 3–5, 18</sub>

## 02 · MSE, and its arithmetic

MSE does what its name says: **difference, square, average**. Squaring stops negative and positive errors cancelling out.

<sub>Source: lecture slides 6–10</sub>

**`mse.py`** — predict the output, then run.

In [ ]:
import torch
import torch.nn as nn
pred   = torch.tensor([2.5, 0.5, 3.0])
target = torch.tensor([3.0, 0.0, 3.0])
loss_fn = nn.MSELoss()
loss = loss_fn(pred, target)
print(loss)

# The math, written as code
manual = ((pred - target) ** 2).mean()
print(manual)               # tensor(0.1667)

<details><summary>▸ Expected output</summary>

```
tensor(0.1667)
tensor(0.1667)
```
</details>

🧪 *Interactive lab in the app:* **MSE by hand** — try it there to see this idea move.

## 03 · Cross-entropy, decomposed

Classification is shaped differently: **one score per class** against a single label. The scores are called **logits**. Not one number against one number — N scores against one index.

<sub>Source: lecture slides 11–16</sub>

**`ce.py`** — predict the output, then run.

In [ ]:
# 3 samples, 4 classes
logits = torch.tensor([
    [2.0, 1.0, 0.1, -1.0],    # sample 0
    [0.5, 2.5, 0.3,  0.2],    # sample 1
    [0.1, 0.2, 3.0,  0.1],    # sample 2
])
labels = torch.tensor([0, 1, 2])      # correct classes
loss_fn = nn.CrossEntropyLoss()
print(loss_fn(logits, labels))

# Sample 0 by hand
logit = torch.tensor([2.0, 1.0, 0.1, -1.0])
probs = torch.softmax(logit, dim=0)
print(probs)
print(-torch.log(probs[0]))

<details><summary>▸ Expected output</summary>

```
tensor(0.3015)
tensor([0.6381, 0.2347, 0.0954, 0.0318])
tensor(0.4493)
```
</details>

🧪 *Interactive lab in the app:* **Softmax → −log → loss** — try it there to see this idea move.

## 04 · The logits trap; binary loss; reduction

`reduction` decides how the per-sample losses collapse — and **`mean` is almost always right**. `[0, 1, 4]` averages to 1.67 and sums to 5. Mean keeps the learning rate independent of batch size; `'none'` lets you weight samples, or find the worst one.

<sub>Source: lecture slides 17, 19–23</sub>

> ⚠️ **Watch out — Never apply softmax before cross-entropy**  
> `CrossEntropyLoss` expects **raw logits**, not probabilities. It applies softmax internally.
>
> If you pass pre-softmaxed probabilities, the loss will be wrong and training will fail quietly — no error, no crash, just a model that learns badly. The same applies to `BCEWithLogitsLoss` — the “WithLogits” suffix is the warning.

**`bce_reduce.py`** — predict the output, then run.

In [ ]:
# Binary classification: spam or not
logit = torch.tensor([1.5])      # raw score
label = torch.tensor([1.0])      # is spam
bce = nn.BCEWithLogitsLoss()
print(bce(logit, label))

pred = torch.tensor([1.0, 2.0, 5.0])
tgt  = torch.tensor([1.0, 3.0, 3.0])
print(nn.MSELoss(reduction='mean')(pred, tgt))
print(nn.MSELoss(reduction='sum')(pred, tgt))
print(nn.MSELoss(reduction='none')(pred, tgt))

<details><summary>▸ Expected output</summary>

```
tensor(0.2014)
tensor(1.6667)
tensor(5.)
tensor([0., 1., 4.])
```
</details>

## 05 · From GD to Adam

Gradients say which way. The **optimizer takes the step**. In Lecture 08 you wrote the rule by hand: `param − lr × grad`. Each optimizer below fixes a problem the previous one had.

<sub>Source: lecture slides 25–32</sub>

**`optimizers.py`** — predict the output, then run.

In [ ]:
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01,
                            momentum=0.9)   # 90% of velocity carries over

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

🧪 *Interactive lab in the app:* **SGD vs momentum vs Adam** — try it there to see this idea move.

## 06 · The two-line API, learning rate and weight decay

Every optimizer has the same interface: **`zero_grad()` → `step()`**. Swap SGD for Adam and nothing else in the loop changes. Every line of the Lecture 03 loop is now explained.

<sub>Source: lecture slides 33–40</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
x = torch.linspace(0, 2, 20).unsqueeze(1)
y = 3 * x + 1 + 0.3 * torch.randn_like(x)
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)


**`train.py`** — predict the output, then run.

In [ ]:
for epoch in range(200):
    pred = model(x)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()   # ❶ clear old gradients
    loss.backward()         # ❷ compute new gradients
    optimizer.step()        # ❸ update parameters

> ⚠️ **Watch out — The order matters**  
> `zero_grad()` → `backward()` → `step()`.
>
> Call `step()` before `backward()` and parameters are updated with stale (or zero) gradients. Skip `zero_grad()` and gradients accumulate and the model diverges.

**`adam_wd.py`** — predict the output, then run.

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4)

## The rule — worth remembering

**Loss functions — the six rules**

1. Regression (predict a number) → `MSELoss`. Classification (pick a class) → `CrossEntropyLoss`.
2. MSE = `((pred - target) ** 2).mean()`. No magic.
3. CrossEntropy = softmax + negative log. Expects raw logits, never pre-softmaxed probabilities.
4. Binary classification → `BCEWithLogitsLoss`. Also expects raw logits.
5. Default `reduction='mean'` is almost always correct.
6. The loss function defines what the model is trying to do. Choose it first, choose it carefully.

**Optimizers — the six rules**

1. An optimizer updates parameters using gradients: `zero_grad()` → `backward()` → `step()`.
2. SGD: simple, reliable. Add `momentum=0.9` for smoother convergence.
3. Adam: adaptive per-parameter rates. Default for most work. Start with `lr=1e-3`.
4. Loss exploding → reduce lr. Loss stuck → increase lr or switch optimizers.
5. `weight_decay` adds regularization — penalizes large weights.
6. The optimizer is a one-line swap. Everything else in the training loop stays the same.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Find the worst prediction

Create logits for 5 samples and 3 classes, with labels `[0, 2, 1, 0, 2]`. Compute `CrossEntropyLoss` with `reduction='none'`. Which sample has the highest loss? Print its logits — is the correct class the highest-scoring one?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`losses = nn.CrossEntropyLoss(reduction='none')(logits, labels)` gives one loss per sample; `losses.argmax()` finds the worst. For that sample the correct class is typically *not* the top score — the model put its probability elsewhere.
</details>

### Exercise 02 · Ch 16 · Ex 3

Apply softmax to a logit vector, confirm probabilities sum to 1, compute `-log(prob[correct])` and compare to `CrossEntropyLoss`.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

For `[2.0, 1.0, 0.1, -1.0]` with class 0: softmax → `[0.6381, 0.2347, 0.0954, 0.0318]` (sum 1.0), −log(0.6381) = 0.4493 — identical to `nn.CrossEntropyLoss()(logit.unsqueeze(0), torch.tensor([0]))`.
</details>

### Exercise 03 · Ch 16 · Ex 4

Apply softmax before `CrossEntropyLoss` — try it, compare to the correct version, explain why it’s wrong.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The loss comes out different (for the sample above about 1.05 instead of 0.45) because softmax is applied twice. Probabilities are squashed into [0, 1], so the second softmax makes them nearly uniform; gradients shrink and the model learns badly — with no error to warn you.
</details>

### Exercise 04 · Ch 17 · Ex 1

Train the Lecture 03 model 3 ways: SGD (lr=0.1), SGD+momentum (lr=0.1, momentum=0.9), Adam (lr=0.01). Print loss every 50 epochs.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

SGD and SGD + momentum both reach loss 0.0667 and recover y ≈ 3.29x + 0.81. Adam at `lr=0.01` does not: after 200 epochs it is still at loss ≈ 2.68 (y ≈ 0.98x + 2.32). Adam’s steps are roughly the size of the learning rate, so 0.01 per step is far too slow to travel from the random start — try `lr=0.1`. Only the optimizer line changes; the loop is identical.
</details>

### Exercise 05 · Ch 17 · Ex 5

Put `step()` before `backward()`. What happens to the loss? Explain why.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Each `step()` now uses the gradients from the *previous* iteration (or none at all on the first), right after they have been zeroed — so with the standard `zero_grad → step → backward` order the parameters never change and the loss stays flat.
</details>

### Interview-style

(a) When would you use MSE vs cross-entropy, and why does `CrossEntropyLoss` expect raw logits?

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

MSE is for regression, where the target is a continuous number and the loss should grow with the squared distance from it. Cross-entropy is for classification, where the model outputs one score per class and the loss measures how much probability landed on the correct one. `CrossEntropyLoss` takes raw logits because it applies softmax internally (in a numerically stable combined form); passing probabilities would apply softmax twice and silently corrupt the loss.
</details>

### Interview-style

(b) Difference between SGD and Adam, and when to choose one over the other? Three sentences each.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

SGD applies one rule to every parameter: subtract the learning rate times the gradient, computed on a mini-batch. It is simple and reliable, and with `momentum=0.9` it smooths its path through narrow valleys. Reach for SGD with momentum where the research community has found it works better.

Adam keeps running averages of each parameter’s gradient and squared gradient, which effectively gives every parameter its own adaptive step size. It works well out of the box at `lr=1e-3` across a wide range of problems. It is the optimizer to start with — Adam first, questions later.
</details>